# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainkhan006/Flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in my slice is one page in March 2026: one content_hash_id (with its client_hash_id) after I roll up daily rows for month=2026-03. It is not one page-day, and it is not the starter CSV’s 90-day snapshot.

The warehouse stores page-days (report_date × client × content). I aggregate those March days so the grain matches Lane 1.

Time window: 1 March 2026 through 31 March 2026 (month=2026-03 only). I do not use June 2026 or fact_content_daily_performance_sample.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Tables I use: fact_content_daily_performance (March partition), dim_content (join on content_hash_id), dim_clients (join on client_hash_id). I list real column names with DESCRIBE before I treat any of them as a feature.

Label / proxy: March visibility — the sum of daily gsc_impressions for that page in March. That is what I score against. It is not a feature.

Feature (five, knowable at the decision moment):

March average GSC position (gsc_avg_position rolled up) — knowable then because it is the position measurement for those days, not a copy of March impressions.
Count of March days where the gsc_data_available IS TRUE — knowable then because it only records whether Search Console was measuring, not the impression total.
March position spread (how much daily position moves inside the month) — knowable then because it comes only from the daily position series.
word_count from dim_content is knowable then because it is a page attribute, not March’s impression outcome.
Client GSC history start (dim_clients.gsc_data_start, or days from that date to March) — knowable then because when that client’s GSC history began is not March’s page-level impressions.
Label / proxy: March summed gsc_impressions (above). March CTR is not a kept feature; I only add it for the leak demo, then delete it.

Context: client_hash_id, content_hash_id, and other hashes on dim_content (url_hash_id, keyword_hash_id if I see them). Join, group, split — never as model features.

Excluded: fact_content_query_90d. Its grain is client × content × query over a fixed 90-day window, not March. Joining it would mix two clocks.

I am also not using the June _sample table to invent labels, and I am not treating hash IDs as signals.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I prove three things on March 2026 only, not on the full warehouse.

First I roll daily rows up to one page (client_hash_id + content_hash_id). Then I check that this rolled-up table has no duplicate pages. If that probe returned rows, my grain would still be page-days, which is not the contract.

Second I print how many pages that slice has, and the first and last report_date that went into the rollup. Those dates should sit inside March 2026.

Third I count March page-days, then count the same rows where gsc_data_available IS TRUE. I use IS TRUE on purpose: NULL is not false, so = TRUE or NOT flag would quietly keep or drop the wrong days.

The rolled-up March table has 331,437 pages and no duplicate page keys, so the grain is one page in March 2026.

Those pages are built from 9,841,378 page-days spanning 1 March through 31 March 2026. I am not using June or the sample table.

Only 3,611,061 of those page-days have Search Console actually measuring (gsc_data_available IS TRUE). The other 6,230,317 days are unmeasured.

The frame has 331,437 pages and no duplicate keys, same grain as the contract.
About 154,699 pages were never measured in March (gsc_data_available never is true). Those pages have no average position. I leave that null. I do not call it position 0. Position spread is null for 168,020 pages: unmeasured pages plus pages with only one measured day. Spread needs at least two measured days.

word_count is missing for 107,429 pages, and GSC history days for 7,067. Those are holes in the dimensions, not March impression totals in disguise.

marchImpressions stays the label. It is not one of the five features.

I first tried March CTR as the leak. Spearman went from 0.8402 to 0.8398. A rate does not encode impression volume, so that was the wrong disguise.

I then added log1p(marchImpressions), which is the label, logged. Spearman rose to 0.8912. That jump is the leak. It is not a better ranking model.

I dropped the logged column. The number I keep is 0.8402, on 110,403 complete-case pages only. CTR and log impressions are not features.

In [3]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb

hfToken = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)",
    [hfToken],
)

rel = "hf://datasets/FlyRank/internship-warehouse"
tables = {
    "dim_clients": f"read_parquet('{rel}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{rel}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{rel}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{rel}/fact_content_query_90d.parquet')",
}

for name, src in tables.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [4]:
factMarch = (
    f"read_parquet('{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet')"
)

print("=== daily fact (March partition) ===")
factSchema = con.sql(f"DESCRIBE SELECT * FROM {factMarch} LIMIT 0").df()
print(factSchema.to_string(index=False))

print("\n=== GSC-related columns on the daily fact ===")
nameCol = factSchema.columns[0]
print(
    factSchema[
        factSchema[nameCol].astype(str).str.contains("gsc", case=False)
    ].to_string(index=False)
)

print("\n=== dim_content ===")
contentSchema = con.sql(
    f"DESCRIBE SELECT * FROM {tables['dim_content']} LIMIT 0"
).df()
print(contentSchema.to_string(index=False))

=== daily fact (March partition) ===
             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None  

In [5]:
print("building the March page table...")

con.sql(f"""
CREATE OR REPLACE TABLE marchPages AS
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(*) AS dayCount,
    MIN(report_date) AS firstDay,
    MAX(report_date) AS lastDay
FROM {factMarch}
GROUP BY client_hash_id, content_hash_id
""")

print("query 1 — grain (duplicate pages should be empty)")
grainDupes = con.sql("""
SELECT client_hash_id, content_hash_id, COUNT(*) AS n
FROM marchPages
GROUP BY client_hash_id, content_hash_id
HAVING COUNT(*) > 1
""").df()
print(grainDupes)
print(f"duplicate page rows: {len(grainDupes)}")

print("\nquery 2 — how big is the slice, and which dates fed it")
print(
    con.sql("""
    SELECT
        COUNT(*) AS pageRows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS distinctPages
    FROM marchPages
    """).df().to_string(index=False)
)
print(
    con.sql(f"""
    SELECT
        MIN(report_date) AS firstReportDate,
        MAX(report_date) AS lastReportDate,
        COUNT(*) AS marchDayRows
    FROM {factMarch}
    """).df().to_string(index=False)
)

print("\nquery 3 — GSC measured days (IS TRUE, not = TRUE)")
print(
    con.sql(f"""
    SELECT
        COUNT(*) AS marchDayRows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS measuredDayRows,
        COUNT(*) - COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS notMeasuredDayRows
    FROM {factMarch}
    """).df().to_string(index=False)
)

building the March page table...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

query 1 — grain (duplicate pages should be empty)
Empty DataFrame
Columns: [client_hash_id, content_hash_id, n]
Index: []
duplicate page rows: 0

query 2 — how big is the slice, and which dates fed it
 pageRows  distinctPages
   331437         331437
firstReportDate lastReportDate  marchDayRows
     2026-03-01     2026-03-31       9841378

query 3 — GSC measured days (IS TRUE, not = TRUE)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 marchDayRows  measuredDayRows  notMeasuredDayRows
      9841378          3611061             6230317


In [6]:
print("collapsing dim_content to one row per page...")

con.sql(f"""
CREATE OR REPLACE TABLE contentWord AS
SELECT
    client_hash_id,
    content_hash_id,
    ANY_VALUE(word_count) AS wordCount
FROM {tables["dim_content"]}
GROUP BY client_hash_id, content_hash_id
""")

print("building the five-feature frame (label kept to the side)...")

con.sql(f"""
CREATE OR REPLACE TABLE marchFeatures AS
SELECT
    d.client_hash_id,
    d.content_hash_id,
    AVG(d.gsc_avg_position) FILTER (
        WHERE d.gsc_data_available IS TRUE
    ) AS avgPosition,
    COUNT(*) FILTER (
        WHERE d.gsc_data_available IS TRUE
    ) AS measuredDayCount,
    STDDEV_SAMP(d.gsc_avg_position) FILTER (
        WHERE d.gsc_data_available IS TRUE
    ) AS positionSpread,
    w.wordCount,
    DATE_DIFF('day', cl.gsc_data_start, DATE '2026-03-01') AS gscHistoryDays,
    SUM(d.gsc_impressions) AS marchImpressions
FROM {factMarch} AS d
LEFT JOIN contentWord AS w
    ON w.client_hash_id = d.client_hash_id
    AND w.content_hash_id = d.content_hash_id
LEFT JOIN {tables["dim_clients"]} AS cl
    ON cl.client_hash_id = d.client_hash_id
GROUP BY
    d.client_hash_id,
    d.content_hash_id,
    w.wordCount,
    cl.gsc_data_start
""")

print("grain check — page rows should still be 331,437 with no duplicates")
print(
    con.sql("""
    SELECT
        COUNT(*) AS pageRows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS distinctPages
    FROM marchFeatures
    """).df().to_string(index=False)
)

print("\ncolumns in the frame (only the five named features plus the label)")
print(con.sql("DESCRIBE marchFeatures").df().to_string(index=False))

print("\npreview (hashes are context, not features)")
print(
    con.sql("""
    SELECT
        avgPosition,
        measuredDayCount,
        positionSpread,
        wordCount,
        gscHistoryDays,
        marchImpressions
    FROM marchFeatures
    LIMIT 5
    """).df().to_string(index=False)
)

print("\nnulls — positionSpread is null when a page has fewer than two measured days")
print(
    con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE avgPosition IS NULL) AS avgPositionNulls,
        COUNT(*) FILTER (WHERE measuredDayCount = 0) AS neverMeasured,
        COUNT(*) FILTER (WHERE positionSpread IS NULL) AS positionSpreadNulls,
        COUNT(*) FILTER (WHERE wordCount IS NULL) AS wordCountNulls,
        COUNT(*) FILTER (WHERE gscHistoryDays IS NULL) AS gscHistoryNulls
    FROM marchFeatures
    """).df().to_string(index=False)
)

collapsing dim_content to one row per page...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

building the five-feature frame (label kept to the side)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

grain check — page rows should still be 331,437 with no duplicates
 pageRows  distinctPages
   331437         331437

columns in the frame (only the five named features plus the label)
     column_name column_type null  key default extra
  client_hash_id     VARCHAR  YES None    None  None
 content_hash_id     VARCHAR  YES None    None  None
     avgPosition      DOUBLE  YES None    None  None
measuredDayCount      BIGINT  YES None    None  None
  positionSpread      DOUBLE  YES None    None  None
       wordCount      BIGINT  YES None    None  None
  gscHistoryDays      BIGINT  YES None    None  None
marchImpressions     HUGEINT  YES None    None  None

preview (hashes are context, not features)
 avgPosition  measuredDayCount  positionSpread  wordCount  gscHistoryDays  marchImpressions
    4.394234                31        2.619436       <NA>             383            1140.0
    7.209549                31        2.442255       2123             383            6523.0
    6.481453      

In [7]:
from sklearn.linear_model import LinearRegression
import pandas as pd

print("building the scoring table (complete cases only)...")

scoreFrame = con.sql(f"""
SELECT
    f.avgPosition,
    f.measuredDayCount,
    f.positionSpread,
    f.wordCount,
    f.gscHistoryDays,
    f.marchImpressions,
    k.marchClicks,
    k.marchClicks * 1.0 / f.marchImpressions AS marchCtr
FROM marchFeatures AS f
INNER JOIN (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS marchClicks
    FROM {factMarch}
    GROUP BY client_hash_id, content_hash_id
) AS k
    ON k.client_hash_id = f.client_hash_id
    AND k.content_hash_id = f.content_hash_id
WHERE f.avgPosition IS NOT NULL
    AND f.positionSpread IS NOT NULL
    AND f.wordCount IS NOT NULL
    AND f.gscHistoryDays IS NOT NULL
    AND f.marchImpressions > 0
""").df()

print(f"pages in this check: {len(scoreFrame):,}")

featureCols = [
    "avgPosition",
    "measuredDayCount",
    "positionSpread",
    "wordCount",
    "gscHistoryDays",
]
label = scoreFrame["marchImpressions"]

def spearmanFromFit(featureDf, label):
    model = LinearRegression()
    model.fit(featureDf, label)
    pred = pd.Series(model.predict(featureDf), index=featureDf.index)
    return pred.corr(label, method="spearman")

honestScore = spearmanFromFit(scoreFrame[featureCols], label)
print(f"honest Spearman (five features only): {honestScore:.4f}")

leakedScore = spearmanFromFit(scoreFrame[featureCols + ["marchCtr"]], label)
print(f"leaked Spearman (five features + March CTR): {leakedScore:.4f}")
print(
    f"the jump is {leakedScore - honestScore:.4f}"
)

scoreFrame = scoreFrame.drop(columns=["marchCtr", "marchClicks"])
print(f" {honestScore:.4f}")
print("columns left:", list(scoreFrame.columns))

building the scoring table (complete cases only)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages in this check: 110,403
honest Spearman (five features only): 0.8402
leaked Spearman (five features + March CTR): 0.8398
the jump is -0.0003
 0.8402
columns left: ['avgPosition', 'measuredDayCount', 'positionSpread', 'wordCount', 'gscHistoryDays', 'marchImpressions']


In [8]:
import numpy as np

print("trap take 2 — leak the label in disguise (log impressions)")

scoreFrame["leakedLogImpressions"] = np.log1p(scoreFrame["marchImpressions"])

leakedScore = spearmanFromFit(
    scoreFrame[featureCols + ["leakedLogImpressions"]],
    label,
)
print(f"honest Spearman (five features only): {honestScore:.4f}")
print(f"leaked Spearman (five features + log impressions): {leakedScore:.4f}")
print(
    f"the jump is {leakedScore - honestScore:.4f} — that is the leak, not a better model"
)

scoreFrame = scoreFrame.drop(columns=["leakedLogImpressions"])
print("dropped leakedLogImpressions")
print(f"I keep the honest Spearman: {honestScore:.4f}")
print("columns left:", list(scoreFrame.columns))

trap take 2 — leak the label in disguise (log impressions)
honest Spearman (five features only): 0.8402
leaked Spearman (five features + log impressions): 0.8912
the jump is 0.0510 — that is the leak, not a better model
dropped leakedLogImpressions
I keep the honest Spearman: 0.8402
columns left: ['avgPosition', 'measuredDayCount', 'positionSpread', 'wordCount', 'gscHistoryDays', 'marchImpressions']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice cannot tell a missing Search Console day from a day with no search demand.

In March 2026 I observe 9,841,378 page-days. Only 3,611,061 have gsc_data_available IS TRUE. The other 6,230,317 were not measured. After I roll up to pages, 154,699 of 331,437 pages never have a measured day, so they have no average position.

I do not treat those rows as zero impressions or position 0. Unmeasured is a hole in coverage, not a visibility outcome. Any ranking I score on this slice is decision-support for pages Search Console actually saw, not a claim about every page-day in March.

The 0.8402 Spearman is the same issue in miniature: it uses 110,403 complete-case pages only.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.